In [16]:
0# =========================
# @title 1. FULL COLAB SETUP - FLEXIBLE DOWNLOAD MODES
# =========================

import os
import shutil
import subprocess

# =========================
# SETTINGS
# =========================

COLAB_PARENT = "/content"
PROJECT_NAME = "TAPETUM"
PROJECT_ROOT = os.path.join(COLAB_PARENT, PROJECT_NAME)

GDRIVE_FOLDER_ID = "13ayyEC3V1wWdX3AXdfL8y7VqnL8eTPFT"
GITHUB_URL = "https://github.com/muratdelen/TAPETUM.git"

# =========================
# DOWNLOAD SOURCE OPTIONS
# =========================

# "driveTAPETUM" -> Download TAPETUM folder directly into /content/TAPETUM
# "drive"        -> Download Drive folder into /content (folder name from Drive)
# "github"       -> Clone TAPETUM repo into /content/TAPETUM
# "auto"         -> Try driveTAPETUM first, fallback to GitHub
DOWNLOAD_SOURCE = "driveTAPETUM"

# =========================
# CLEAN OLD PROJECT
# =========================

if os.path.exists(PROJECT_ROOT):
    print("Removing old TAPETUM project...")
    shutil.rmtree(PROJECT_ROOT)

# =========================
# INSTALL GDOWN
# =========================

subprocess.run("pip install -q gdown", shell=True, check=True)

# =========================
# DOWNLOAD FUNCTIONS
# =========================
def download_drive_tapetum_fast():
    import os

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = "/content/drive/MyDrive/TAPETUM"
    PROJECT_ROOT = "/content/TAPETUM"

    os.system(f"rm -rf {PROJECT_ROOT}")
    os.makedirs(PROJECT_ROOT, exist_ok=True)

    print("Copying RetinexTapetum...")
    os.system(f"cp -r {DRIVE_ROOT}/RetinexTapetum {PROJECT_ROOT}/")

    print("Copying RetinexFormer...")
    os.system(f"cp -r {DRIVE_ROOT}/RetinexFormer {PROJECT_ROOT}/")

    print("Copying RetinexNet...")
    os.system(f"cp -r {DRIVE_ROOT}/RetinexNet {PROJECT_ROOT}/")

    print("Copying Zero-DCE...")
    os.system(f"cp -r {DRIVE_ROOT}/Zero-DCE {PROJECT_ROOT}/")

    print("Copying LoLv2...")
    os.system(f"cp -r {DRIVE_ROOT}/LoLv2 {PROJECT_ROOT}/")

    print("Copying datasets...")
    os.system(f"cp -r {DRIVE_ROOT}/datasets {PROJECT_ROOT}/")


    print("DONE.")

def download_drive_tapetum():
    """
    Mount Google Drive and copy TAPETUM folder
    from /content/drive/MyDrive/TAPETUM -> /content/TAPETUM
    """

    print("Mounting Google Drive...")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_TAPETUM_PATH = "/content/drive/MyDrive/TAPETUM"

    if not os.path.exists(DRIVE_TAPETUM_PATH):
        raise RuntimeError("Drive TAPETUM folder not found at /MyDrive/TAPETUM")

    print("Copying TAPETUM from Drive to Colab local disk...")

    try:
        shutil.copytree(DRIVE_TAPETUM_PATH, PROJECT_ROOT)
    except Exception as e:
        print("Standard copy failed, retrying with system copy:", e)
        os.system(f"cp -r {DRIVE_TAPETUM_PATH} {PROJECT_ROOT}")

    # Verify
    if not os.path.exists(PROJECT_ROOT) or len(os.listdir(PROJECT_ROOT)) == 0:
        raise RuntimeError("Copy failed or empty project.")

    print("driveTAPETUM COPY SUCCESS")


def download_drive_generic():
    """
    Download Google Drive folder into /content (keeps original folder name)
    """

    print("Downloading Drive folder into /content (original name)...")

    os.chdir(COLAB_PARENT)

    cmd = (
        f'gdown --folder '
        f'"https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}" '
        f'--remaining-ok'
    )

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("drive download failed.")

    print("drive SUCCESS")


def download_github():
    """
    Clone TAPETUM repo
    """

    print("Cloning from GitHub -> /content/TAPETUM")

    os.chdir(COLAB_PARENT)

    result = subprocess.run(
        f'git clone "{GITHUB_URL}" "{PROJECT_ROOT}"',
        shell=True,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("GitHub clone failed.")

    print("GitHub SUCCESS")


# =========================
# MAIN LOGIC
# =========================

if DOWNLOAD_SOURCE == "driveTAPETUM":
    download_drive_tapetum_fast()

elif DOWNLOAD_SOURCE == "drive":
    download_drive_generic()

elif DOWNLOAD_SOURCE == "github":
    download_github()

elif DOWNLOAD_SOURCE == "auto":
    try:
        download_drive_tapetum_fast()
    except Exception as e:
        print("Drive failed:", e)
        print("Switching to GitHub...")
        download_github()

else:
    raise ValueError("Invalid DOWNLOAD_SOURCE")

# =========================
# FINAL CHECK
# =========================

print("\nPROJECT STATUS")

if os.path.exists(PROJECT_ROOT):
    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("\nContents:")
    for f in os.listdir(PROJECT_ROOT)[:30]:
        print("-", f)
else:
    print("PROJECT_ROOT not created or empty.")

Removing old TAPETUM project...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying RetinexTapetum...
Copying RetinexFormer...
Copying RetinexNet...
Copying Zero-DCE...
Copying LoLv2...
Copying datasets...
DONE.

PROJECT STATUS
PROJECT_ROOT: /content/TAPETUM

Contents:
- RetinexTapetum
- LoLv2
- RetinexFormer
- RetinexNet
- datasets
- Zero-DCE


In [23]:
# ============================================================
# @title 2. COLAB - FINAL UPDATED 4 MODEL METRIC TABLE
# Method | Params(M) | FPS | PSNR | SSIM | LPIPS
# Models: RetinexTapetum, RetinexNet, Zero-DCE, RetinexFormer
# ============================================================

import os, sys, time, glob, importlib.util
import numpy as np
from PIL import Image

!pip -q install lpips scikit-image pandas tabulate openpyxl pyyaml

import torch
import torch.nn as nn
import pandas as pd
import lpips
import yaml
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# =========================
# ROOT
# =========================
DRIVE_ROOT = "/content/TAPETUM"

# =========================
# DATASET PATHS
# =========================
DATA_LOW = os.path.join(
    DRIVE_ROOT,
    "datasets/LoLv2/LOL-v2/Real_captured/Test/Low"
)

DATA_GT = os.path.join(
    DRIVE_ROOT,
    "datasets/LoLv2/LOL-v2/Real_captured/Test/Normal"
)

# =========================
# RESULT IMAGE PATHS
# =========================
MODEL_RESULT_DIRS = {
    "RetinexTapetum": os.path.join(DRIVE_ROOT, "LoLv2/RetinexTapetum/results/Test"),
    "RetinexNet": os.path.join(DRIVE_ROOT, "LoLv2/RetinexNet/result/Test"),
    "Zero-DCE": os.path.join(DRIVE_ROOT, "LoLv2/ZeroDCE/results/Test"),
    "RetinexFormer": os.path.join(DRIVE_ROOT, "LoLv2/RetinexFormer/results/Test"),
}

# =========================
# MODEL PATHS
# =========================
RETINEXTAPETUM_PATH = os.path.join(DRIVE_ROOT, "RetinexTapetum")
RETINEXTAPETUM_WEIGHT = os.path.join(
    DRIVE_ROOT,
    "LoLv2/RetinexTapetum/checkpoints/best.pth"
)

ZERODCE_MODEL = os.path.join(
    DRIVE_ROOT,
    "Zero-DCE/Zero-DCE_code/model.py"
)
ZERODCE_WEIGHT = os.path.join(
    DRIVE_ROOT,
    "Zero-DCE/model/Epoch99.pth"
)

RETINEXFORMER_PATH = os.path.join(
    DRIVE_ROOT,
    "RetinexFormer"
)
RETINEXFORMER_CONFIG = os.path.join(
    RETINEXFORMER_PATH,
    "Options/RetinexFormer_LOL_v2_real.yml"
)
RETINEXFORMER_WEIGHT = os.path.join(
    RETINEXFORMER_PATH,
    "pretrained_weights/LOL_v2_real.pth"
)

RETINEXNET_PATH = os.path.join(DRIVE_ROOT, "RetinexNet")
RETINEXNET_MODEL = os.path.join(RETINEXNET_PATH, "model.py")

# =========================
# OPTIONAL REAL COMPLEXITY FALLBACK
# This is not manual; it reads your previously measured file if present.
# =========================
COMPLEXITY_XLSX = os.path.join(
    DRIVE_ROOT,
    "model_complexity_all_models_updated.xlsx"
)

# =========================
# SAVE PATHS
# =========================
SUMMARY_DIR = os.path.join(DRIVE_ROOT, "paper_metrics_summary")
os.makedirs(SUMMARY_DIR, exist_ok=True)

CSV_PATH = os.path.join(SUMMARY_DIR, "four_model_metrics_final_updated.csv")
LATEX_PATH = os.path.join(SUMMARY_DIR, "four_model_metrics_final_updated_latex.txt")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ============================================================
# IMAGE HELPERS
# ============================================================

IMG_EXTS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

def list_images(folder):
    files = []
    if not os.path.isdir(folder):
        return files

    for ext in IMG_EXTS:
        files.extend(glob.glob(os.path.join(folder, f"*{ext}")))

    return sorted(files)


def normalize_name(path):
    name = os.path.splitext(os.path.basename(path))[0]
    name = name.replace("_S", "")
    name = name.replace("_s", "")
    name = name.replace("_enhanced", "")
    name = name.replace("_result", "")
    return name


def read_rgb(path):
    img = Image.open(path).convert("RGB")
    return np.array(img).astype(np.float32) / 255.0


def resize_to_gt(pred, gt):
    if pred.shape[:2] == gt.shape[:2]:
        return pred

    pred_img = Image.fromarray((np.clip(pred, 0, 1) * 255).astype(np.uint8))
    pred_img = pred_img.resize((gt.shape[1], gt.shape[0]), Image.BICUBIC)

    return np.array(pred_img).astype(np.float32) / 255.0


def image_to_lpips_tensor(img_np):
    t = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float()
    t = t * 2.0 - 1.0
    return t.to(DEVICE)

# ============================================================
# METRICS
# ============================================================

lpips_model = lpips.LPIPS(net="alex").to(DEVICE)
lpips_model.eval()

def evaluate_result_folder(model_name, pred_dir):
    gt_files = list_images(DATA_GT)
    pred_files = list_images(pred_dir)

    gt_map = {normalize_name(p): p for p in gt_files}
    pred_map = {normalize_name(p): p for p in pred_files}

    common = sorted(set(gt_map.keys()) & set(pred_map.keys()))

    if len(common) == 0:
        print(f"[WARNING] No matched files for {model_name}")
        return {
            "matched": 0,
            "psnr": None,
            "ssim": None,
            "lpips": None
        }

    psnr_list = []
    ssim_list = []
    lpips_list = []

    with torch.no_grad():
        for key in common:
            gt = read_rgb(gt_map[key])
            pr = read_rgb(pred_map[key])
            pr = resize_to_gt(pr, gt)

            psnr_val = peak_signal_noise_ratio(gt, pr, data_range=1.0)
            ssim_val = structural_similarity(gt, pr, channel_axis=2, data_range=1.0)

            gt_t = image_to_lpips_tensor(gt)
            pr_t = image_to_lpips_tensor(pr)
            lpips_val = lpips_model(pr_t, gt_t).item()

            psnr_list.append(psnr_val)
            ssim_list.append(ssim_val)
            lpips_list.append(lpips_val)

    return {
        "matched": len(common),
        "psnr": float(np.mean(psnr_list)),
        "ssim": float(np.mean(ssim_list)),
        "lpips": float(np.mean(lpips_list))
    }

# ============================================================
# MODEL HELPERS
# ============================================================

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6


def forward_only(model, x):
    y = model(x)

    if isinstance(y, dict):
        if "enhanced" in y:
            return y["enhanced"]
        return list(y.values())[0]

    if isinstance(y, (tuple, list)):
        return y[0]

    return y


def measure_fps(model, size=256, warmup=10, runs=50):
    model = model.to(DEVICE).eval()
    x = torch.randn(1, 3, size, size).to(DEVICE)

    with torch.no_grad():
        for _ in range(warmup):
            _ = forward_only(model, x)

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start = time.time()

        for _ in range(runs):
            _ = forward_only(model, x)

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        end = time.time()

    return runs / (end - start)


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for k in [
            "model",
            "state_dict",
            "model_state_dict",
            "net",
            "params",
            "params_ema"
        ]:
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]

    return ckpt


def clean_state_dict(sd):
    cleaned = {}

    for k, v in sd.items():
        nk = k

        for prefix in ["module.", "model.", "net.", "generator."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]

        cleaned[nk] = v

    return cleaned

# ============================================================
# FALLBACK FROM REAL COMPLEXITY XLSX
# ============================================================

def read_real_complexity_from_xlsx(model_name):
    if not os.path.isfile(COMPLEXITY_XLSX):
        return None, None, "complexity xlsx not found"

    dfc = pd.read_excel(COMPLEXITY_XLSX)

    if "Model" not in dfc.columns:
        return None, None, "complexity xlsx missing Model column"

    row = dfc[dfc["Model"].astype(str) == model_name]

    if len(row) == 0:
        return None, None, "model not found in complexity xlsx"

    row = row.iloc[0]

    if str(row.get("Status", "")).lower() != "ok":
        return None, None, str(row.get("Errors", "complexity failed"))

    params = float(row["Params_Million"])
    fps = float(row["FPS_256x256"])

    return params, fps, "REAL_FROM_COMPLEXITY_XLSX"

# ============================================================
# RETINEXTAPETUM LOADER
# ============================================================

def load_retinextapetum_real():
    sys.path.insert(0, RETINEXTAPETUM_PATH)

    from model import RetinexTapetumModel

    model = RetinexTapetumModel(
        base=32,
        lambda_init=0.0,
        lambda_max=1.68
    ).to(DEVICE)

    if not os.path.isfile(RETINEXTAPETUM_WEIGHT):
        raise FileNotFoundError(f"Checkpoint not found: {RETINEXTAPETUM_WEIGHT}")

    checkpoint = torch.load(RETINEXTAPETUM_WEIGHT, map_location=DEVICE)

    if "model" in checkpoint:
        model.load_state_dict(checkpoint["model"], strict=True)
    else:
        model.load_state_dict(checkpoint, strict=True)

    model.eval()
    return model

# ============================================================
# ZERO-DCE LOADER
# ============================================================

def load_zerodce_real():
    if not os.path.isfile(ZERODCE_MODEL):
        raise FileNotFoundError(f"Zero-DCE model.py not found: {ZERODCE_MODEL}")

    sys.path.insert(0, os.path.dirname(ZERODCE_MODEL))

    spec = importlib.util.spec_from_file_location(
        "zerodce_model",
        ZERODCE_MODEL
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    if hasattr(mod, "enhance_net_nopool"):
        model = mod.enhance_net_nopool()
    elif hasattr(mod, "enhance_net"):
        model = mod.enhance_net()
    else:
        raise RuntimeError("Zero-DCE model class not found")

    if not os.path.isfile(ZERODCE_WEIGHT):
        raise FileNotFoundError(f"Checkpoint not found: {ZERODCE_WEIGHT}")

    ckpt = torch.load(ZERODCE_WEIGHT, map_location=DEVICE)
    sd = clean_state_dict(extract_state_dict(ckpt))

    model.load_state_dict(sd, strict=False)
    model.to(DEVICE).eval()

    return model

# ============================================================
# RETINEXFORMER LOADER - UPDATED FINAL
# ============================================================

def load_retinexformer_real():
    sys.path.insert(0, RETINEXFORMER_PATH)

    arch_file = os.path.join(
        RETINEXFORMER_PATH,
        "basicsr/models/archs/RetinexFormer_arch.py"
    )

    if not os.path.isfile(arch_file):
        raise FileNotFoundError(f"RetinexFormer arch not found: {arch_file}")

    spec = importlib.util.spec_from_file_location(
        "retinexformer_arch_real",
        arch_file
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    if not hasattr(mod, "RetinexFormer"):
        raise RuntimeError("RetinexFormer class not found in arch file")

    RetinexFormer = mod.RetinexFormer

    # Safe default for RetinexFormer LOL-v2 real.
    kwargs = {
        "in_channels": 3,
        "out_channels": 3,
        "n_feat": 40,
        "stage": 1,
        "num_blocks": [1, 2, 2],
    }

    # Prefer YAML config if it exists.
    if os.path.isfile(RETINEXFORMER_CONFIG):
        with open(RETINEXFORMER_CONFIG, "r") as f:
            opt = yaml.safe_load(f)

        if isinstance(opt, dict) and "network_g" in opt:
            net_cfg = opt["network_g"].copy()
            net_cfg.pop("type", None)

            # Only keep keys that constructor accepts.
            sig = inspect.signature(RetinexFormer.__init__)
            valid_args = set(sig.parameters.keys())

            for k, v in net_cfg.items():
                if k in valid_args:
                    kwargs[k] = v

    # Final filtering by constructor signature.
    sig = inspect.signature(RetinexFormer.__init__)
    valid_args = set(sig.parameters.keys())

    kwargs = {
        k: v for k, v in kwargs.items()
        if k in valid_args
    }

    print("RetinexFormer final constructor args:", kwargs)

    model = RetinexFormer(**kwargs).to(DEVICE)

    if not os.path.isfile(RETINEXFORMER_WEIGHT):
        raise FileNotFoundError(f"Checkpoint not found: {RETINEXFORMER_WEIGHT}")

    ckpt = torch.load(RETINEXFORMER_WEIGHT, map_location=DEVICE)

    if isinstance(ckpt, dict):
        if "params" in ckpt:
            sd = ckpt["params"]
        elif "params_ema" in ckpt:
            sd = ckpt["params_ema"]
        elif "state_dict" in ckpt:
            sd = ckpt["state_dict"]
        elif "model" in ckpt:
            sd = ckpt["model"]
        else:
            sd = ckpt
    else:
        sd = ckpt

    sd = clean_state_dict(sd)

    load_info = model.load_state_dict(sd, strict=False)

    print("RetinexFormer missing keys:", len(load_info.missing_keys))
    print("RetinexFormer unexpected keys:", len(load_info.unexpected_keys))

    model.eval()
    return model

# ============================================================
# RETINEXNET LOADER
# TensorFlow legacy model usually needs sess, so Params/FPS can remain N/A.
# ============================================================

def load_retinexnet_real():
    if not os.path.isfile(RETINEXNET_MODEL):
        raise FileNotFoundError(f"RetinexNet model.py not found: {RETINEXNET_MODEL}")

    sys.path.insert(0, RETINEXNET_PATH)

    spec = importlib.util.spec_from_file_location(
        "retinexnet_model",
        RETINEXNET_MODEL
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    candidates = [
        "RetinexNet",
        "DecomNet",
        "RelightNet",
        "Model",
        "Net"
    ]

    for cls_name in candidates:
        if hasattr(mod, cls_name):
            cls = getattr(mod, cls_name)

            try:
                model = cls()

                if isinstance(model, nn.Module):
                    model.to(DEVICE).eval()
                    return model

            except Exception:
                pass

    raise RuntimeError("RetinexNet is legacy TensorFlow or requires sess")

# ============================================================
# PARAMS / FPS
# ============================================================

def get_params_fps(model_name):
    try:
        if model_name == "RetinexTapetum":
            model = load_retinextapetum_real()
            return count_params(model), measure_fps(model), "REAL_MEASURED_NOW"

        if model_name == "Zero-DCE":
            model = load_zerodce_real()
            return count_params(model), measure_fps(model), "REAL_MEASURED_NOW"

        if model_name == "RetinexFormer":
            model = load_retinexformer_real()
            return count_params(model), measure_fps(model), "REAL_MEASURED_NOW"

        if model_name == "RetinexNet":
            model = load_retinexnet_real()
            return count_params(model), measure_fps(model), "REAL_MEASURED_NOW"

    except Exception as e:
        params, fps, note = read_real_complexity_from_xlsx(model_name)

        if params is not None and fps is not None:
            return params, fps, note + f" | direct_load_error: {e}"

        return None, None, f"{model_name} ERROR: {e} | fallback: {note}"

    return None, None, "UNKNOWN_MODEL"

# ============================================================
# MAIN
# ============================================================

rows = []

print("\n==============================")
print("EVALUATION START")
print("==============================")

for model_name, result_dir in MODEL_RESULT_DIRS.items():
    print(f"\n--- {model_name} ---")
    print("Result dir:", result_dir)

    if not os.path.isdir(result_dir):
        print("Result folder not found.")
        metrics = {
            "matched": 0,
            "psnr": None,
            "ssim": None,
            "lpips": None
        }
    else:
        metrics = evaluate_result_folder(model_name, result_dir)

    params_m, fps, note = get_params_fps(model_name)

    rows.append({
        "Method": model_name,
        "Params (M)↓": params_m,
        "FPS↑": fps,
        "PSNR↑": metrics["psnr"],
        "SSIM↑": metrics["ssim"],
        "LPIPS↓": metrics["lpips"],
        "Matched": metrics["matched"],
        "Note": note
    })

df = pd.DataFrame(rows)

display_df = df.copy()

for col in [
    "Params (M)↓",
    "FPS↑",
    "PSNR↑",
    "SSIM↑",
    "LPIPS↓"
]:
    display_df[col] = display_df[col].apply(
        lambda x: "N/A" if x is None or pd.isna(x) else round(float(x), 4)
    )

print("\n==============================")
print("FINAL TABLE")
print("==============================")

display(display_df[
    [
        "Method",
        "Params (M)↓",
        "FPS↑",
        "PSNR↑",
        "SSIM↑",
        "LPIPS↓",
        "Matched",
        "Note"
    ]
])

df.to_csv(CSV_PATH, index=False)

latex_df = display_df[
    [
        "Method",
        "Params (M)↓",
        "FPS↑",
        "PSNR↑",
        "SSIM↑",
        "LPIPS↓"
    ]
]

latex_text = latex_df.to_latex(index=False, escape=False)

with open(LATEX_PATH, "w", encoding="utf-8") as f:
    f.write(latex_text)

print("\nSaved CSV  :", CSV_PATH)
print("Saved LaTeX:", LATEX_PATH)

print("\nLaTeX Table:")
print(latex_text)

DEVICE: cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth

EVALUATION START

--- RetinexTapetum ---
Result dir: /content/TAPETUM/LoLv2/RetinexTapetum/results/Test

--- RetinexNet ---
Result dir: /content/TAPETUM/LoLv2/RetinexNet/result/Test

--- Zero-DCE ---
Result dir: /content/TAPETUM/LoLv2/ZeroDCE/results/Test

--- RetinexFormer ---
Result dir: /content/TAPETUM/LoLv2/RetinexFormer/results/Test
RetinexFormer final constructor args: {'in_channels': 3, 'out_channels': 3, 'n_feat': 40, 'stage': 1, 'num_blocks': [1, 2, 2]}
RetinexFormer missing keys: 0
RetinexFormer unexpected keys: 0

FINAL TABLE


,Method,Params (M)↓,FPS↑,PSNR↑,SSIM↑,LPIPS↓,Matched,Note
0,RetinexTapetum,0.5247,85.0329,19.2473,0.7734,0.3669,100,REAL_MEASURED_NOW
1,RetinexNet,N/A,N/A,15.9504,0.6524,0.4128,100,RetinexNet ERROR: RetinexNet is legacy TensorF...
2,Zero-DCE,0.0794,229.1018,17.9993,0.5778,0.3126,100,REAL_MEASURED_NOW
3,RetinexFormer,1.6057,23.3112,22.7940,0.8386,0.1707,100,REAL_MEASURED_NOW



Saved CSV  : /content/TAPETUM/paper_metrics_summary/four_model_metrics_final_updated.csv
Saved LaTeX: /content/TAPETUM/paper_metrics_summary/four_model_metrics_final_updated_latex.txt

LaTeX Table:
\begin{tabular}{lllrrr}
\toprule
Method & Params (M)↓ & FPS↑ & PSNR↑ & SSIM↑ & LPIPS↓ \\
\midrule
RetinexTapetum & 0.524700 & 85.032900 & 19.247300 & 0.773400 & 0.366900 \\
RetinexNet & N/A & N/A & 15.950400 & 0.652400 & 0.412800 \\
Zero-DCE & 0.079400 & 229.101800 & 17.999300 & 0.577800 & 0.312600 \\
RetinexFormer & 1.605700 & 23.311200 & 22.794000 & 0.838600 & 0.170700 \\
\bottomrule
\end{tabular}



In [2]:
# =========================
# @title 2. PATH CHECK
# =========================
# =========================
# 0. PROJECT PATH CONFIG
# =========================

import os
import sys
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = "/content/TAPETUM"

assert os.path.exists(PROJECT_ROOT), f"Klasör bulunamadı: {PROJECT_ROOT}"

# =========================
# DATASET PATHS
# =========================

DATA_ROOT = os.path.join(
    PROJECT_ROOT,
    "datasets/LoLv2/LOL-v2/Real_captured"
)

DATA_LOW = os.path.join(DATA_ROOT, "Test/Low")
DATA_GT  = os.path.join(DATA_ROOT, "Test/Normal")

# =========================
# OUTPUT ROOT
# =========================

OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "OUTPUTS")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# =========================
# RetinexTapetum
# =========================

RETINEXTAPETUM_PATH   = os.path.join(PROJECT_ROOT, "RetinexTapetum")
RETINEXTAPETUM_CONFIG = os.path.join(RETINEXTAPETUM_PATH, "config.py")
RETINEXTAPETUM_MODEL  = os.path.join(RETINEXTAPETUM_PATH, "model.py")
RETINEXTAPETUM_TEST   = os.path.join(RETINEXTAPETUM_PATH, "test.py")
RETINEXTAPETUM_TRAIN  = os.path.join(RETINEXTAPETUM_PATH, "train.py")
RETINEXTAPETUM_CKPT   = os.path.join(RETINEXTAPETUM_PATH, "checkpoints")
RETINEXTAPETUM_WEIGHT = os.path.join(RETINEXTAPETUM_CKPT, "best.pth")

RETINEXTAPETUM_IMAGES = os.path.join(
    PROJECT_ROOT, "LoLv2/RetinexTapetum/result/Test"
)

RETINEXTAPETUM_SAVE_IMAGES = os.path.join(
    OUTPUT_ROOT, "RetinexTapetum"
)

RUN_RETINEXTAPETUM = True

# =========================
# RetinexNet
# =========================

RETINEXNET_PATH   = os.path.join(PROJECT_ROOT, "RetinexNet")
RETINEXNET_CONFIG = None
RETINEXNET_MODEL  = os.path.join(RETINEXNET_PATH, "model.py")
RETINEXNET_TEST   = os.path.join(RETINEXNET_PATH, "main.py")
RETINEXNET_CKPT   = os.path.join(RETINEXNET_PATH, "checkpoint")

RETINEXNET_IMAGES = os.path.join(
    PROJECT_ROOT, "LoLv2/RetinexNet/result/Test"
)

RETINEXNET_SAVE_IMAGES = os.path.join(
    OUTPUT_ROOT, "RetinexNet"
)

RUN_RETINEXNET = False

# =========================
# Zero-DCE
# =========================

ZERODCE_PATH   = os.path.join(PROJECT_ROOT, "Zero-DCE")
ZERODCE_CONFIG = None
ZERODCE_MODEL  = os.path.join(ZERODCE_PATH, "Zero-DCE_code/model.py")
ZERODCE_TEST   = os.path.join(ZERODCE_PATH, "Zero-DCE_code/lowlight_test.py")
ZERODCE_TRAIN  = os.path.join(ZERODCE_PATH, "Zero-DCE_code/lowlight_train.py")
ZERODCE_CKPT   = os.path.join(ZERODCE_PATH, "model/Epoch99.pth")
ZERODCE_WEIGHT = ZERODCE_CKPT

ZERODCE_IMAGES = os.path.join(
    PROJECT_ROOT, "LoLv2/ZeroDCE/result/Test"
)

ZERODCE_SAVE_IMAGES = os.path.join(
    OUTPUT_ROOT, "ZeroDCE"
)

RUN_ZERODCE = False

# =========================
# RetinexFormer
# =========================

RETINEXFORMER_PATH = os.path.join(PROJECT_ROOT, "RetinexFormer")

RETINEXFORMER_CONFIG = os.path.join(
    RETINEXFORMER_PATH,
    "Options/RetinexFormer_LOL_v2_real.yml"
)

RETINEXFORMER_MODEL = os.path.join(
    RETINEXFORMER_PATH,
    "basicsr/models/archs/RetinexFormer_arch.py"
)

RETINEXFORMER_TEST = os.path.join(
    RETINEXFORMER_PATH,
    "Enhancement/test_from_dataset.py"
)

RETINEXFORMER_CKPT = os.path.join(
    RETINEXFORMER_PATH,
    "pretrained_weights/LOL_v2_real.pth"
)

RETINEXFORMER_WEIGHT = RETINEXFORMER_CKPT

RETINEXFORMER_IMAGES = os.path.join(
    PROJECT_ROOT, "LoLv2/RetinexFormer/result/Test"
)

RETINEXFORMER_SAVE_IMAGES = os.path.join(
    OUTPUT_ROOT, "RetinexFormer"
)

RUN_RETINEXFORMER = False

# =========================
# CREATE OUTPUT FOLDERS
# =========================

for p in [
    RETINEXTAPETUM_SAVE_IMAGES,
    RETINEXNET_SAVE_IMAGES,
    ZERODCE_SAVE_IMAGES,
    RETINEXFORMER_SAVE_IMAGES,
]:
    os.makedirs(p, exist_ok=True)

print("PROJECT ROOT:", PROJECT_ROOT)
print("DATA LOW    :", DATA_LOW)
print("DATA GT     :", DATA_GT)
print("OUTPUT ROOT :", OUTPUT_ROOT)
# =========================
# DATA PATHS - COLAB LOCAL
# =========================


DATA_LOW = os.path.join(
    PROJECT_ROOT,
    "datasets/LoLv2/LOL-v2/Real_captured/Test/Low"
)

DATA_GT = os.path.join(
    PROJECT_ROOT,
    "datasets/LoLv2/LOL-v2/Real_captured/Test/Normal"
)
paths_to_check = {
    "DATA_LOW": DATA_LOW,
    "DATA_GT": DATA_GT,

    "RETINEXTAPETUM_PATH": RETINEXTAPETUM_PATH,
    "RETINEXTAPETUM_CONFIG": RETINEXTAPETUM_CONFIG,
    "RETINEXTAPETUM_MODEL": RETINEXTAPETUM_MODEL,
    "RETINEXTAPETUM_TEST": RETINEXTAPETUM_TEST,
    "RETINEXTAPETUM_WEIGHT": RETINEXTAPETUM_WEIGHT,

    "RETINEXNET_PATH": RETINEXNET_PATH,
    "RETINEXNET_MODEL": RETINEXNET_MODEL,
    "RETINEXNET_TEST": RETINEXNET_TEST,
    "RETINEXNET_CKPT": RETINEXNET_CKPT,

    "ZERODCE_PATH": ZERODCE_PATH,
    "ZERODCE_MODEL": ZERODCE_MODEL,
    "ZERODCE_TEST": ZERODCE_TEST,
    "ZERODCE_WEIGHT": ZERODCE_WEIGHT,

    "RETINEXFORMER_PATH": RETINEXFORMER_PATH,
    "RETINEXFORMER_CONFIG": RETINEXFORMER_CONFIG,
    "RETINEXFORMER_TEST": RETINEXFORMER_TEST,
    "RETINEXFORMER_WEIGHT": RETINEXFORMER_WEIGHT,
}

missing = []

for name, path in paths_to_check.items():
    exists = os.path.exists(path) if path is not None else True
    print(f"{'✅' if exists else '❌'} {name}: {path}")
    if not exists:
        missing.append((name, path))

print("\nLOW IMAGE COUNT:", len(os.listdir(DATA_LOW)) if os.path.exists(DATA_LOW) else 0)
print("GT IMAGE COUNT :", len(os.listdir(DATA_GT)) if os.path.exists(DATA_GT) else 0)

if missing:
    print("\n❌\n")
    for name, path in missing:
        print("-", name, "=>", path)
else:
    print("\n✅\n")

PROJECT ROOT: /content/TAPETUM
DATA LOW    : /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Low
DATA GT     : /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Normal
OUTPUT ROOT : /content/TAPETUM/OUTPUTS
✅ DATA_LOW: /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Low
✅ DATA_GT: /content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Normal
✅ RETINEXTAPETUM_PATH: /content/TAPETUM/RetinexTapetum
✅ RETINEXTAPETUM_CONFIG: /content/TAPETUM/RetinexTapetum/config.py
✅ RETINEXTAPETUM_MODEL: /content/TAPETUM/RetinexTapetum/model.py
✅ RETINEXTAPETUM_TEST: /content/TAPETUM/RetinexTapetum/test.py
✅ RETINEXTAPETUM_WEIGHT: /content/TAPETUM/RetinexTapetum/checkpoints/best.pth
✅ RETINEXNET_PATH: /content/TAPETUM/RetinexNet
✅ RETINEXNET_MODEL: /content/TAPETUM/RetinexNet/model.py
✅ RETINEXNET_TEST: /content/TAPETUM/RetinexNet/main.py
✅ RETINEXNET_CKPT: /content/TAPETUM/RetinexNet/checkpoint
✅ ZERODCE_PATH: /content/TAPETUM/Zero-DCE
✅ ZERODCE_MODEL: /content/TAPETUM/Zero

In [13]:
# =========================
# @title 3. Target directory to reset (LoLv2/RetinexTapetum)
# =========================
import shutil, os

# Target directory to reset
TARGET_DIR = "/content/TAPETUM/LoLv2/RetinexTapetum"

# Check if the folder exists before deleting
if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

# Create a clean empty directory
os.makedirs(TARGET_DIR, exist_ok=True)

# Print confirmation message
print("Folder has been reset safely.")

Folder has been reset safely.


In [1]:
# =========================
# @title 3. RUN RETINEXTAPETUM TRAIN IN COLAB
# =========================

import os
import subprocess
import sys

# =========================
# PROJECT PATHS
# =========================

PROJECT_ROOT = "/content/TAPETUM"
RETINEXTAPETUM_PATH = os.path.join(PROJECT_ROOT, "RetinexTapetum")
TRAIN_SCRIPT = os.path.join(RETINEXTAPETUM_PATH, "train.py")

# =========================
# EXPECTED PATHS FROM config.py
# =========================

TRAIN_LOW_DIR = os.path.join(PROJECT_ROOT, "datasets/LoLv2/LOL-v2/Real_captured/Train/Low")
TRAIN_HIGH_DIR = os.path.join(PROJECT_ROOT, "datasets/LoLv2/LOL-v2/Real_captured/Train/Normal")
VAL_LOW_DIR = os.path.join(PROJECT_ROOT, "datasets/LoLv2/LOL-v2/Real_captured/Test/Low")
VAL_HIGH_DIR = os.path.join(PROJECT_ROOT, "datasets/LoLv2/LOL-v2/Real_captured/Test/Normal")

CKPT_DIR = os.path.join(PROJECT_ROOT, "RetinexTapetum/checkpoints")
RESULT_DIR = os.path.join(PROJECT_ROOT, "LoLv2/RetinexTapetum/results/Test")

# =========================
# CREATE OUTPUT FOLDERS
# =========================

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# =========================
# PATH CHECK
# =========================

check_paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "RETINEXTAPETUM_PATH": RETINEXTAPETUM_PATH,
    "train.py": TRAIN_SCRIPT,
    "TRAIN_LOW_DIR": TRAIN_LOW_DIR,
    "TRAIN_HIGH_DIR": TRAIN_HIGH_DIR,
    "VAL_LOW_DIR": VAL_LOW_DIR,
    "VAL_HIGH_DIR": VAL_HIGH_DIR,
    "CKPT_DIR": CKPT_DIR,
    "RESULT_DIR": RESULT_DIR,
}

missing = []

print("===== PATH CHECK =====")

for name, path in check_paths.items():
    exists = os.path.exists(path)
    print(f"{'✅ FOUND' if exists else '❌ MISSING'} | {name}")
    print(path)
    print("-" * 80)

    if not exists:
        missing.append((name, path))

print("Train Low Count :", len(os.listdir(TRAIN_LOW_DIR)) if os.path.exists(TRAIN_LOW_DIR) else 0)
print("Train GT Count  :", len(os.listdir(TRAIN_HIGH_DIR)) if os.path.exists(TRAIN_HIGH_DIR) else 0)
print("Val Low Count   :", len(os.listdir(VAL_LOW_DIR)) if os.path.exists(VAL_LOW_DIR) else 0)
print("Val GT Count    :", len(os.listdir(VAL_HIGH_DIR)) if os.path.exists(VAL_HIGH_DIR) else 0)

if missing:
    raise FileNotFoundError("Some required paths are missing. Fix the folder structure first.")

# =========================
# RUN TRAINING
# =========================

print("\n===== STARTING RETINEXTAPETUM TRAINING =====")

command = [sys.executable, "train.py"]

result = subprocess.run(
    command,
    cwd=RETINEXTAPETUM_PATH,
    text=True
)

if result.returncode != 0:
    raise RuntimeError("RetinexTapetum training failed.")

print("\n===== TRAINING FINISHED =====")
print("Best checkpoint:", os.path.join(CKPT_DIR, "best.pth"))
print("Last checkpoint:", os.path.join(CKPT_DIR, "last.pth"))

===== PATH CHECK =====
✅ FOUND | PROJECT_ROOT
/content/TAPETUM
--------------------------------------------------------------------------------
❌ MISSING | RETINEXTAPETUM_PATH
/content/TAPETUM/RetinexTapetum
--------------------------------------------------------------------------------
❌ MISSING | train.py
/content/TAPETUM/RetinexTapetum/train.py
--------------------------------------------------------------------------------
❌ MISSING | TRAIN_LOW_DIR
/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Train/Low
--------------------------------------------------------------------------------
❌ MISSING | TRAIN_HIGH_DIR
/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Train/Normal
--------------------------------------------------------------------------------
❌ MISSING | VAL_LOW_DIR
/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Low
--------------------------------------------------------------------------------
❌ MISSING | VAL_HIGH_DIR
/content/TAPETUM/datasets/LoLv2/

FileNotFoundError: Some required paths are missing. Fix the folder structure first.

In [14]:
# =========================
# @title 4. RUN RETINEXTAPETUM TEST IN COLAB - CONFIG MATCHED
# =========================

import os
import subprocess
import sys
import shutil

PROJECT_ROOT = "/content/TAPETUM"
RETINEXTAPETUM_PATH = os.path.join(PROJECT_ROOT, "RetinexTapetum")

TEST_SCRIPT = os.path.join(RETINEXTAPETUM_PATH, "test.py")
CONFIG_SCRIPT = os.path.join(RETINEXTAPETUM_PATH, "config.py")

TEST_LOW_DIR = os.path.join(
    PROJECT_ROOT,
    "datasets/LoLv2/LOL-v2/Real_captured/Test/Low"
)

CKPT_DIR = os.path.join(
    PROJECT_ROOT,
    "LoLv2/RetinexTapetum/checkpoints"
)

CKPT_PATH = os.path.join(
    CKPT_DIR,
    "best.pth"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "LoLv2/RetinexTapetum/results/Test"
)

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# If best.pth exists in RetinexTapetum/checkpoints, copy it to the config.py checkpoint path
ALT_CKPT_PATH = os.path.join(
    PROJECT_ROOT,
    "RetinexTapetum/checkpoints/best.pth"
)

if not os.path.exists(CKPT_PATH) and os.path.exists(ALT_CKPT_PATH):
    shutil.copy2(ALT_CKPT_PATH, CKPT_PATH)
    print("Checkpoint copied:")
    print("FROM:", ALT_CKPT_PATH)
    print("TO  :", CKPT_PATH)

# Clear previous outputs
for item in os.listdir(RESULT_DIR):
    item_path = os.path.join(RESULT_DIR, item)
    if os.path.isfile(item_path) or os.path.islink(item_path):
        os.remove(item_path)
    else:
        shutil.rmtree(item_path)

print("Previous test outputs cleared.")

# Path check
check_paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "RETINEXTAPETUM_PATH": RETINEXTAPETUM_PATH,
    "config.py": CONFIG_SCRIPT,
    "test.py": TEST_SCRIPT,
    "TEST_LOW_DIR": TEST_LOW_DIR,
    "CKPT_DIR": CKPT_DIR,
    "BEST_MODEL": CKPT_PATH,
    "RESULT_DIR": RESULT_DIR,
}

missing = []

print("\n===== PATH CHECK =====")

for name, path in check_paths.items():
    exists = os.path.exists(path)
    print(f"{'✅ FOUND' if exists else '❌ MISSING'} | {name}")
    print(path)
    print("-" * 80)

    if not exists:
        missing.append((name, path))

print("Test image count:", len(os.listdir(TEST_LOW_DIR)) if os.path.exists(TEST_LOW_DIR) else 0)

if missing:
    raise FileNotFoundError("Missing paths. Fix before running test.")

# Run test.py with visible output
print("\n===== STARTING RETINEXTAPETUM TEST =====")

result = subprocess.run(
    [sys.executable, "-u", "test.py"],
    cwd=RETINEXTAPETUM_PATH,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("\n===== TEST STDOUT =====")
print(result.stdout)

print("\n===== TEST STDERR =====")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Test failed.")

# Check generated outputs
output_images = [
    f for f in os.listdir(RESULT_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
]

print("\n===== TEST FINISHED =====")
print("Expected result folder:")
print(RESULT_DIR)
print("Output image count:", len(output_images))

if len(output_images) > 0:
    print("First 10 output images:")
    for f in output_images[:10]:
        print("-", f)
else:
    print("No images found in expected RESULT_DIR.")
    print("Searching all result folders under /content/TAPETUM...")

    for dirpath, dirnames, filenames in os.walk(PROJECT_ROOT):
        imgs = [
            f for f in filenames
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
        ]

        if len(imgs) > 0 and "result" in dirpath.lower():
            print(dirpath, "=>", len(imgs))

Checkpoint copied:
FROM: /content/TAPETUM/RetinexTapetum/checkpoints/best.pth
TO  : /content/TAPETUM/LoLv2/RetinexTapetum/checkpoints/best.pth
Previous test outputs cleared.

===== PATH CHECK =====
✅ FOUND | PROJECT_ROOT
/content/TAPETUM
--------------------------------------------------------------------------------
✅ FOUND | RETINEXTAPETUM_PATH
/content/TAPETUM/RetinexTapetum
--------------------------------------------------------------------------------
✅ FOUND | config.py
/content/TAPETUM/RetinexTapetum/config.py
--------------------------------------------------------------------------------
✅ FOUND | test.py
/content/TAPETUM/RetinexTapetum/test.py
--------------------------------------------------------------------------------
✅ FOUND | TEST_LOW_DIR
/content/TAPETUM/datasets/LoLv2/LOL-v2/Real_captured/Test/Low
--------------------------------------------------------------------------------
✅ FOUND | CKPT_DIR
/content/TAPETUM/LoLv2/RetinexTapetum/checkpoints
---------------------

In [ ]:
# =========================
# @title 5. OPTIONAL COPY RESULTS TO DRIVE
# =========================
COLAB_OUT_ROOT ="/content/TAPETUM/LoLv2"
DRIVE_OUT_ROOT ="/content/drive/MyDrive/TAPETUM/OUTPUTS"
SAVE_TO_DRIVE = True

print("\nDONE.")
print("Results saved in Colab:")
print(COLAB_OUT_ROOT)

if SAVE_TO_DRIVE:
    print("\nCopying results to Drive...")

    if os.path.exists(DRIVE_OUT_ROOT):
        shutil.rmtree(DRIVE_OUT_ROOT)

    shutil.copytree(COLAB_OUT_ROOT, DRIVE_OUT_ROOT)

    print("Results also saved to Drive:")
    print(DRIVE_OUT_ROOT)
else:
    print("\nDrive saving disabled.")

In [4]:
# =========================
# @title 6. GOOGLE DRIVE → COLAB COPY WITH RSYNC
# =========================

import os

DRIVE_OUT_ROOT = "/content/drive/MyDrive/TAPETUM/LoLv2"
COLAB_OUT_ROOT = "/content/TAPETUM/LoLv2"

os.makedirs(COLAB_OUT_ROOT, exist_ok=True)

print("Drive → Colab rsync ile kopyalanıyor...")

!rsync -av --delete "$DRIVE_OUT_ROOT/" "$COLAB_OUT_ROOT/"

print("Drive → Colab kopyalama tamamlandı.")

Drive → Colab rsync ile kopyalanıyor...
sending incremental file list
deleting RetinexFormer/result/Test/
deleting RetinexFormer/result/
RetinexFormer/
deleting RetinexTapetum/result/Test/
deleting RetinexTapetum/result/
deleting ZeroDCE/result/Test/
deleting ZeroDCE/result/
RetinexTapetum/
RetinexTapetum/results/Test/00690.png
RetinexTapetum/results/Test/00691.png
RetinexTapetum/results/Test/00692.png
RetinexTapetum/results/Test/00693.png
RetinexTapetum/results/Test/00694.png
RetinexTapetum/results/Test/00695.png
RetinexTapetum/results/Test/00696.png
RetinexTapetum/results/Test/00697.png
RetinexTapetum/results/Test/00698.png
RetinexTapetum/results/Test/00699.png
RetinexTapetum/results/Test/00700.png
RetinexTapetum/results/Test/00701.png
RetinexTapetum/results/Test/00702.png
RetinexTapetum/results/Test/00703.png
RetinexTapetum/results/Test/00704.png
RetinexTapetum/results/Test/00705.png
RetinexTapetum/results/Test/00706.png
RetinexTapetum/results/Test/00707.png
RetinexTapetum/results/T